# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

## Finding 1

**Paper finding:** [actual finding from the FlyRank research paper]

**Methodology question:** What observed outcome or label was used to support this finding, and was that outcome measured in a way that matches the decision being discussed?

I would also check whether the validation design separates the data used for model development from the data used to evaluate the reported result. This would help determine whether the finding is likely to generalize beyond the evaluated sample.

## Finding 2

**Paper finding:** [actual finding from the FlyRank research paper]

**Methodology question:** Does the validation design support the strength of the claim being made? In particular, I would check whether the evaluation avoids overlap between development and evaluation data and whether the validation setup reflects the intended use case.

This is a constructive methodology check rather than a claim that the finding is incorrect.

In [26]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
paper_findings = {
    "Finding 1": "The Anatomy of Growing Content",
    "Finding 2": "AI Model Performance"
}

for finding, title in paper_findings.items():
    print(f"{finding}: {title}")

print("\nMethodology audit questions recorded for both findings.")


Finding 1: The Anatomy of Growing Content
Finding 2: AI Model Performance

Methodology audit questions recorded for both findings.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

## My Model Under an Honest Split

My Week-5 model was initially evaluated using a train/test setup. For this audit, I use a client-grouped split so that pages from the same client do not appear in both training and test data.

This is a more realistic test of whether the model generalizes to clients that were not used during training.

I compare the model with the Week-4 baseline using the same Precision@K metric. The before/after comparison shows whether the model's performance remains useful under the stricter validation design.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [27]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier

!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
%cd flyrank-ml-internship-starter

# Load the starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Observed target
df["is_declining_label"] = (
    df["trend_direction"].str.lower().eq("down")
).astype(int)

print("Rows:", len(df))
print("Clients:", df["client_id"].nunique())

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 5.01 MiB/s, done.
Resolving deltas: 100% (153/153), done.
/content/flyrank-ml-internship-starter/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Rows: 30000
Clients: 32


In [28]:
# Same feature set used for the Week-5 model
features = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "ctr",
    "avg_position",
    "engagement_rate",
    "scroll_rate",
    "search_volume",
    "competition",
    "word_count"
]

X = df[features].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

y = df["is_declining_label"]

print("Features used:", len(features))
print(features)

Features used: 14
['content_age_days', 'days_since_last_update', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'search_volume', 'competition', 'word_count']


In [29]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=df["client_id"]
    )
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

train_clients = set(df.iloc[train_idx]["client_id"])
test_clients = set(df.iloc[test_idx]["client_id"])

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Client overlap:", len(train_clients & test_clients))

Training rows: 22885
Test rows: 7115
Training clients: 24
Test clients: 8
Client overlap: 0


In [30]:
model = RandomForestClassifier(
    n_estimators=200,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_score = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


In [37]:
test_df = df.iloc[test_idx].copy()

test_df["baseline_score"] = (
    (test_df["days_since_last_update"] >= 180).astype(int)
    *
    (test_df["impressions_90d"] >= 500).astype(int)
    *
    test_df["impressions_90d"]
)

baseline_score = test_df["baseline_score"].values

In [32]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    top_k = np.asarray(labels)[order[:k]]
    return top_k.mean()


results = []

for k in [10, 20, 50]:

    baseline_precision = precision_at_k(
        baseline_score,
        y_test.values,
        k
    )

    model_precision = precision_at_k(
        model_score,
        y_test.values,
        k
    )

    results.append({
        "K": k,
        "Baseline_Precision": baseline_precision,
        "Model_Precision": model_precision,
        "Difference": model_precision - baseline_precision
    })

honest_results = pd.DataFrame(results)

print("Honest client-grouped validation:")
display(honest_results)

Honest client-grouped validation:


,K,Baseline_Precision,Model_Precision,Difference
0,10,0.60,0.40,-0.2
1,20,0.50,0.40,-0.1
2,50,0.62,0.52,-0.1


## Leakage Audit

I reviewed the final feature set for variables that contain the outcome directly or are derived from the outcome.

The model does not use `trend_direction`, `trend_pct`, or `is_declining_label` as features. `trend_direction` is used only to construct the observed evaluation target, while `trend_pct` is excluded because it is used in deriving the trend label.

The remaining features are intended to represent observable content, search, and engagement signals rather than the outcome itself.

This audit does not prove that every possible source of temporal leakage has been eliminated, so the result is treated as directional evidence rather than causal proof.

In [33]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


leakage_candidates = [
    "trend_pct",
    "trend_direction",
    "is_declining_label"
]

leakage_audit = pd.DataFrame({
    "feature": leakage_candidates,
    "used_as_model_feature": [
        feature in features
        for feature in leakage_candidates
    ]
})

display(leakage_audit)

,feature,used_as_model_feature
0,trend_pct,False
1,trend_direction,False
2,is_declining_label,False


In [34]:
print("Final model feature set:")
for feature in features:
    print("-", feature)

print("\nLeakage candidates used as features:",
      [f for f in leakage_candidates if f in features])

Final model feature set:
- content_age_days
- days_since_last_update
- impressions_90d
- clicks_90d
- pageviews_90d
- sessions_90d
- users_90d
- ctr
- avg_position
- engagement_rate
- scroll_rate
- search_volume
- competition
- word_count

Leakage candidates used as features: []


In [35]:
assert "is_declining_label" not in features
assert "trend_direction" not in features
assert "trend_pct" not in features

print("Leakage check passed: target and outcome-derived fields are excluded.")

Leakage check passed: target and outcome-derived fields are excluded.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Claim Rewrite

### Stronger claim I might have made

"The Random Forest predicts which pages will benefit from being refreshed."

### Safer claim supported by the evidence

"The Random Forest produced a directional ranking of content pages using observed historical search, engagement, and content signals. Under the evaluated client-grouped split, its performance was measured against the Week-4 baseline using Precision@K. The results provide decision-support evidence about which pages may warrant review; they do not establish that refreshing a recommended page will cause higher rankings, traffic, or engagement."

The safer wording reflects what was actually measured and avoids making a causal claim that the available data cannot establish.

## 4. Claim rewrite

### Original stronger claim

"The Random Forest predicts which pages will benefit from being refreshed."

### Revised claim

"The Random Forest provides a directional ranking of content pages using observed historical search, engagement, and content signals. Its performance was measured against the Week-4 baseline using Precision@K and was also evaluated using a client-grouped split. These results provide decision-support evidence about which pages may warrant editorial review, but they do not prove that refreshing a recommended page will cause higher rankings, traffic, or engagement."

The revised claim is limited to what was actually measured. It does not claim causal impact or guarantee future search performance.

## 4. Claim rewrite

### Original stronger claim

"The Random Forest predicts which pages will benefit from being refreshed."

### Revised claim

"The Random Forest provides a directional ranking of content pages using observed historical search, engagement, and content signals. In the evaluated data, the model did not outperform the Week-4 baseline on Precision@10, Precision@20, or Precision@50. The client-grouped validation produced the same measured values in this run. These results provide decision-support evidence about which pages may warrant editorial review, but they do not prove that refreshing a recommended page will cause higher rankings, traffic, or engagement."

The revised claim is limited to what was actually measured. It does not claim causal impact or guarantee future search performance.


In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.